# Fine-tuning BERT cho Masked Language Model

Trong bài này, chúng ta sẽ fine-tune một mô hình BERT đã được huấn luyện trước để thực hiện  Masked Language Model (MLM).

## 1. Load dataset

Tập dữ liệu được sử dụng trong bài này là `imdb`, tập dữ liệu chứa các bài đánh giá phim

In [1]:
from datasets import load_dataset

dataset = load_dataset("imdb")

print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and

## 2. Load tokenizer

Tokenizer sẽ biến câu thành các token mà BERT có thể xử lý.

In [2]:
from transformers import AutoTokenizer

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

## 3. Tokenize dataset

Ta chỉ tokenize cột `text`.

Không dùng cột `label` của IMDB vì nhiệm vụ hiện tại không phải phân loại cảm xúc.

In [3]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Bỏ các cột gốc không cần thiết
remove_columns = [col for col in tokenized_dataset["train"].column_names 
                  if col not in ["input_ids", "attention_mask"]]

tokenized_dataset = tokenized_dataset.remove_columns(remove_columns)

print(tokenized_dataset["train"][0].keys())

dict_keys(['input_ids', 'attention_mask'])


## 4. Tạo Data Collator cho Masked Language Model

`DataCollatorForLanguageModeling` sẽ tự động che ngẫu nhiên một phần token trong câu.

Tham số `mlm_probability=0.15` nghĩa là khoảng 15% token sẽ được chọn để huấn luyện dự đoán lại.


In [4]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

## 5. Load mô hình Masked Language Model

Ta dùng `AutoModelForMaskedLM` để mô hình học dự đoán token bị che.

In [5]:
from transformers import AutoModelForMaskedLM

model = AutoModelForMaskedLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 6. Chia tập train/eval nhỏ để chạy nhanh

Dữ liệu IMDB khá lớn, nên ở đây chỉ lấy một phần nhỏ để demo.

In [6]:
small_train_dataset = tokenized_dataset["train"].shuffle(seed=42).select(range(2000))
small_eval_dataset = tokenized_dataset["test"].shuffle(seed=42).select(range(500))

print(small_train_dataset)
print(small_eval_dataset)

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 2000
})
Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 500
})


## 7. Cấu hình huấn luyện

Với masked-language model, loss được tính dựa trên khả năng dự đoán đúng các token đã bị che.

In [7]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results_masked_lm",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    learning_rate=5e-5,
    weight_decay=0.01,
    report_to="none"
)

## 8. Tạo Trainer

Trainer sẽ dùng `data_collator` để tự tạo input bị mask và label tương ứng.

In [8]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    data_collator=data_collator
)

## 9. Huấn luyện và đánh giá mô hình

In [9]:
from transformers import TrainerCallback

trainer.callback_handler.callbacks = [
    cb for cb in trainer.callback_handler.callbacks
    if cb.__class__.__name__ in ["DefaultFlowCallback"]
]

In [10]:
trainer.train()
eval_result = trainer.evaluate()
print(eval_result)

/home/jupyter-iec2024iot08/.local/lib/python3.12/site-packages/torch/nn/parallel/data_parallel.py:37: UserWarning: 
    There is an imbalance between your GPUs. You may want to exclude GPU 1 which
    has less than 75% of the memory or cores of GPU 0. You can do so by setting
    the device_ids argument to DataParallel, or by setting the CUDA_VISIBLE_DEVICES
    environment variable.
  warnings.warn(
/home/jupyter-iec2024iot08/.local/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/jupyter-iec2024iot08/.local/lib/python3.12/site-packages/torch/nn/parallel/data_parallel.py:37: UserWarning: 
    There is an imbalance between your GPUs. You may want to exclude GPU 1 which
    has less than 75% of the memory or cores of GPU 0. You can do so by setting
    the device_ids argument to DataParallel, or by setting the CUDA_VISIBLE_DEVICES
    environment variable.
  warnings.warn(
/home/jupyter-iec2024iot08/.local/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


{'eval_loss': 4.166268825531006, 'eval_runtime': 7.9419, 'eval_samples_per_second': 62.957, 'eval_steps_per_second': 4.029, 'epoch': 1.0}


## 10. Dự đoán token bị che

Ta đưa vào câu có `[MASK]`, mô hình sẽ gợi ý các token phù hợp nhất.

In [11]:
from transformers import pipeline

fill_mask = pipeline(
    "fill-mask",
    model="./saved_masked_lm",
    tokenizer="./saved_masked_lm"
)

examples = [
    "This movie was very [MASK].",
    "I would [MASK] this film to my friends.",
    "I don't really [MASK] the plot of the movie.",
]

for text in examples:
    print("Input:", text)
    predictions = fill_mask(text)
    for pred in predictions[:5]:
        print(pred["sequence"], "| score:", round(pred["score"], 4))
    print("-" * 80)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Input: This movie was very [MASK].
this movie was very good. | score: 0.634
this movie was very entertaining. | score: 0.0677
this movie was very interesting. | score: 0.0302
this movie was very successful. | score: 0.0301
this movie was very bad. | score: 0.0299
--------------------------------------------------------------------------------
Input: I would [MASK] this film to my friends.
i would recommend this film to my friends. | score: 0.6216
i would show this film to my friends. | score: 0.0601
i would give this film to my friends. | score: 0.0556
i would present this film to my friends. | score: 0.0354
i would leave this film to my friends. | score: 0.0176
--------------------------------------------------------------------------------
Input: I don't really [MASK] the plot of the movie.
i don ' t really understand the plot of the movie. | score: 0.3637
i don ' t really know the plot of the movie. | score: 0.3351
i don ' t really like the plot of the movie. | score: 0.1498
i don '

## Kết luận

Qua bài này, chúng ta đã thấy cách fine-tune BERT cho nhiệm vụ Masked Language Model. 